# Notebook 4: A Kit of Moves with Nothing Wasted (basis and dimension)

**Predict, then run.** Run the two setup cells first.

**Setup.** The next cell is the drawing library, `vecdraw`. It is short, and you are allowed to read it and change it.

In [ ]:
%%writefile vecdraw.py
"""
vecdraw - a tiny helper for drawing vectors on graph paper.

READ ME! Nothing in here is magic. Every function is a few lines of
NumPy and matplotlib that you could have written yourself.
Change something, run the cell again, and see what happens.
"""
import numpy as np
import matplotlib.pyplot as plt

# Colors that are easy to tell apart (also for color-blind readers).
COLORS = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9"]

_size = 6     # how far the graph paper reaches in each direction
_count = 0    # how many arrows we have drawn so far (picks the next color)


def new_grid(size=6):
    """Start a fresh sheet of graph paper that runs from -size to +size."""
    global _size, _count
    _size = size
    _count = 0
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(-size, size)
    ax.set_ylim(-size, size)
    ax.set_aspect("equal")                  # one step right = one step up
    if size <= 12:                          # small paper: a line at every whole number
        ax.set_xticks(range(-size, size + 1))
        ax.set_yticks(range(-size, size + 1))
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color="black", linewidth=1)   # the x-axis
    ax.axvline(0, color="black", linewidth=1)   # the y-axis
    return ax


def _paper():
    """Give back the sheet we are drawing on. Make one if there is none."""
    if plt.get_fignums():
        return plt.gca()
    return new_grid(_size)


def _next_color():
    """Pick the next color from the list, starting over when we run out."""
    global _count
    color = COLORS[_count % len(COLORS)]
    _count = _count + 1
    return color


def draw_vector(v, start=(0, 0), color=None, label=None, label_at="tip"):
    """Draw the vector v as an arrow that begins at start.

    The label goes just past the tip, or beside the middle of the arrow
    if you say label_at="middle".
    Gives back the tip of the arrow (start + v), so the next arrow
    can begin where this one ended.
    """
    ax = _paper()
    v = np.array(v, dtype=float)
    start = np.array(start, dtype=float)
    if color is None:
        color = _next_color()
    ax.quiver(start[0], start[1], v[0], v[1],
              angles="xy", scale_units="xy", scale=1,   # draw at true size
              color=color, width=0.008, zorder=3)
    tip = start + v
    if label is not None:
        length = np.linalg.norm(v)
        if length == 0:
            spot = tip + np.array([0.0, _size * 0.07])
        elif label_at == "middle":
            sideways = np.array([-v[1], v[0]]) / length   # v turned a quarter turn
            spot = start + v / 2 + sideways * _size * 0.07
        else:
            spot = tip + v / length * _size * 0.07        # just past the tip
        ax.text(spot[0], spot[1], label, color=color,
                fontsize=13, fontweight="bold", ha="center", va="center")
    return tip


def draw_point(p, label=None, color="black"):
    """Draw a dot at the point p, with an optional name next to it."""
    ax = _paper()
    ax.plot(p[0], p[1], "o", color=color, markersize=8, zorder=4)
    if label is not None:
        ax.text(p[0] + _size * 0.04, p[1] + _size * 0.04, label,
                color=color, fontsize=12)


def draw_path(steps, start=(0, 0), labels=None):
    """Draw a list of vectors tip-to-tail, like a trip with several legs.

    Gives back the place where the trip ends.
    """
    place = np.array(start, dtype=float)
    for i, step in enumerate(steps):
        name = labels[i] if labels is not None else None
        place = draw_vector(step, start=place, label=name, label_at="middle")
    return place


def draw_dots(points, color="gray", size=18):
    """Draw many points at once. points is a list of [x, y] pairs."""
    ax = _paper()
    points = np.array(points, dtype=float)
    ax.scatter(points[:, 0], points[:, 1], s=size, color=color, zorder=2)


def draw_skewed_grid(b1, b2, color="#E69F00"):
    """Draw the slanted graph paper you get from the two moves b1 and b2."""
    ax = _paper()
    n = 3 * _size                           # enough lines to cover the sheet
    b1 = np.array(b1, dtype=float)
    b2 = np.array(b2, dtype=float)
    for k in range(-n, n + 1):
        # a line in the b1 direction, shifted over by k copies of b2
        begin = k * b2 - n * b1
        end = k * b2 + n * b1
        ax.plot([begin[0], end[0]], [begin[1], end[1]], color=color, linewidth=1, alpha=0.7)
        # a line in the b2 direction, shifted over by k copies of b1
        begin = k * b1 - n * b2
        end = k * b1 + n * b2
        ax.plot([begin[0], end[0]], [begin[1], end[1]], color=color, linewidth=1, alpha=0.7)


def angle_between(a, b):
    """The angle between vectors a and b, in degrees."""
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    cosine = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    cosine = np.clip(cosine, -1, 1)     # guard against tiny rounding errors
    degrees = np.degrees(np.arccos(cosine))
    return round(float(degrees), 2)     # two decimal places is plenty


def show():
    """Show the picture."""
    plt.show()

In [ ]:
import numpy as np
import importlib, vecdraw
importlib.reload(vecdraw)      # pick up any changes you make to the library
from vecdraw import *

## Warm-up: same skills, new numbers

On paper first.

1. Are `(1, 2)` and `(2, 4)` dependent or independent?
2. Are `(1, 2)` and `(3, 1)` dependent or independent?
3. Finish the function below so it returns `True` when the spot is `(0, 0)`.

In [ ]:
a = np.array([1, 2])
b = np.array([3, 1])
k = np.array([2, 4])
m = np.array([4, 3])


def is_home(spot):
    return np.allclose(spot, 0)


print(is_home(2 * a - 1 * k))
print(is_home(2 * a - 1 * b))

## Part 1: Too few, too many, just right

Here is a function that counts the ways to reach a target with whole-number weights from -6 to 6. It takes two things as input: a **list of moves** and a **target**.

Read it. `len(moves)` is how many moves are in the list.

In [ ]:
def count_ways(moves, target):
    ways = 0
    if len(moves) == 1:
        for c in range(-6, 7):
            if np.array_equal(c * moves[0], target):
                ways = ways + 1
    if len(moves) == 2:
        for c in range(-6, 7):
            for d in range(-6, 7):
                if np.array_equal(c * moves[0] + d * moves[1], target):
                    ways = ways + 1
    if len(moves) == 3:
        for c in range(-6, 7):
            for d in range(-6, 7):
                for e in range(-6, 7):
                    if np.array_equal(c * moves[0] + d * moves[1] + e * moves[2], target):
                        ways = ways + 1
    return ways

The charger is at `(7, 4)`. Predict the number of ways for each kit of moves. Then run.

In [ ]:
charger = np.array([7, 4])

print("only a:      ", count_ways([a], charger))
print("a and k:     ", count_ways([a, k], charger))
print("a and b:     ", count_ways([a, b], charger))
print("a, b and m:  ", count_ways([a, b, m], charger))

**Talk about it.**

- Which kits cannot reach the charger at all? What is wrong with them?
- Which kit reaches it in many ways? What is wrong with that one?
- Which kit is just right?

**Your turn.** Try three other chargers. Does the "just right" kit always give exactly one way? (If it gives zero, think about Part 6 of Notebook 1.)

## Part 2: The address of a point

With the kit `a`, `b` there is exactly one pair of weights for each spot. Call that pair the spot's **address**.

Searching is slow. NumPy can find the address directly. First put the moves side by side in a table. A table of numbers like this is called a **2D array**.

In [ ]:
kit = np.column_stack([a, b])
print(kit)
print(kit.shape)        # rows, columns

Each **column** of the table is one move. `np.linalg.solve` finds the weights that reach the target.

In [ ]:
print(np.linalg.solve(kit, charger))

Wrap it in a function so it is easy to use again. Then check the answer by rebuilding the trip.

In [ ]:
def address(target, move1, move2):
    kit = np.column_stack([move1, move2])
    return np.linalg.solve(kit, target)


weights = address(charger, a, b)
print(weights)
print(weights[0] * a + weights[1] * b)      # do we get the charger back?

## Part 3: Slanted graph paper

Normal graph paper is built from the moves `(1, 0)` and `(0, 1)`. The moves `a` and `b` build their own slanted graph paper.

Find the charger at `(7, 4)` on the slanted paper. Start at home. How many `a` steps and how many `b` steps? Does it match the address?

In [ ]:
new_grid(8)
draw_skewed_grid(a, b)
draw_vector(a, label="a")
draw_vector(b, label="b")
draw_point(charger, "charger")
show()

**One spot, many addresses.** The charger did not move. Only the kit of moves changed. Predict each address before you run.

In [ ]:
e1 = np.array([1, 0])
e2 = np.array([0, 1])
p = np.array([1, 1])
q = np.array([1, -1])

print("with (1,0) and (0,1): ", address(charger, e1, e2))
print("with a and b:         ", address(charger, a, b))
print("with (1,1) and (1,-1):", address(charger, p, q))

**Your turn.** Draw the slanted paper for `p` and `q` and find the charger on it by counting steps.

In [ ]:
new_grid(8)
draw_skewed_grid(p, q)
draw_vector(p, label="p")
draw_vector(q, label="q")
draw_point(charger, "charger")
show()

## Part 4: When there is no address

What happens if you ask for an address using the broken kit `a` and `k`? Predict.

`try` means "attempt this". If it crashes, Python runs the `except` part instead of stopping.

In [ ]:
try:
    print(address(charger, a, k))
except np.linalg.LinAlgError:
    print("NumPy gave up: these two moves cannot give every spot an address.")

## The names for this

A kit of moves is a **basis** when it is

- big enough to reach everywhere (its span is the whole space), and
- has nothing wasted (the moves are independent).

With a basis, every spot has **exactly one** address.

The number of moves in a basis is called the **dimension**. A line needs 1 move. Flat paper needs 2. The room you are sitting in needs 3.

## Your turn: is it a basis for flat paper?

Decide on paper and say why. Then test with `address` or `count_ways`.

1. `(1, 0)` and `(0, 1)`
2. `(1, 2)` and `(2, 4)`
3. `(1, 1)` and `(1, -1)`
4. `(1, 0)`, `(0, 1)` and `(1, 1)`
5. `(2, 3)` all by itself

In [ ]:
try:
    print(address(np.array([5, 1]), np.array([1, 1]), np.array([1, -1])))
except np.linalg.LinAlgError:
    print("not a basis")

## Check yourself

On paper.

1. Find the address of `(5, 5)` using the basis `a = (1, 2)`, `b = (3, 1)`.
2. Find the address of `(5, 1)` using the basis `(1, 1)`, `(1, -1)`.
3. Give a basis for the line `y = 3x`. What is the dimension of that line?
4. Why can three moves never be a basis for flat paper?
5. A feature vector in your ML class has 4 numbers. How many moves would a basis for those vectors need?

---
# The Deep End

## Deep 1: The dot-product shortcut

For the basis `p = (1, 1)`, `q = (1, -1)` there is a way to get the address with **no solving at all**: just dot products.

Check that it gives the right address for a few spots. Then try the same shortcut with `a` and `b`. It fails. What is different about `p` and `q`? (Notebook 0, Part 5.)

*This leads to: perpendicular bases and projection (Module 12).*

In [ ]:
spot = np.array([5, 1])

shortcut = np.array([np.dot(spot, p) / np.dot(p, p),
                     np.dot(spot, q) / np.dot(q, q)])
print("shortcut:", shortcut)
print("solve:   ", address(spot, p, q))

## Deep 2: The address translator

A friend gives you an address in the `a`, `b` kit: "2 of a and 1 of b". You only speak `p`, `q`. Write a function that translates any `a`, `b` address into a `p`, `q` address.

Test it: translate, rebuild the spot both ways, and make sure it is the same spot.

*This leads to: change of basis, and multiplying matrices (Modules 5 and 8).*

In [ ]:
def translate(ab_address):
    spot = ab_address[0] * a + ab_address[1] * b
    return spot        # finish me: return the p, q address of this spot


print(translate(np.array([2, 1])))

## Deep 3: Pictures are vectors

A tiny 2-by-2 picture is 4 numbers, one brightness per pixel. So it is a vector with 4 numbers.

Here is a basis for all 2-by-2 pictures: four pictures with a single pixel lit. Build a checkerboard as a linear combination of them. Build a picture that gets brighter from left to right.

How many moves would a basis need for the 28-by-28 pictures your ML class uses?

In [ ]:
import matplotlib.pyplot as plt

pixel1 = np.array([[1, 0], [0, 0]])
pixel2 = np.array([[0, 1], [0, 0]])
pixel3 = np.array([[0, 0], [1, 0]])
pixel4 = np.array([[0, 0], [0, 1]])

picture = 1.0 * pixel1 + 0.3 * pixel2 + 0.6 * pixel3 + 0.0 * pixel4    # change the weights
print(picture)

plt.figure(figsize=(2, 2))
plt.imshow(picture, cmap="gray", vmin=0, vmax=1)
plt.axis("off")
plt.show()

## Deep 4: A basis for a tilted floor

In 3D, look at all the spots where `x + y + z = 0`. That is a flat sheet through home.

Find two moves that stay on the sheet and are independent. How do you know they reach **every** spot on the sheet? What is the dimension of the sheet?

Check a few trips: do they stay on the sheet?

In [ ]:
f1 = np.array([1, -1, 0])
f2 = np.array([0, 0, 0])       # change me

spot = 2 * f1 + 3 * f2
print(spot, " x + y + z =", spot.sum())

## Deep 5: Draw the slanted paper yourself

Print the source of `draw_skewed_grid` and read it. Then write your own version, using matplotlib directly, that draws the `a` lines in one color and the `b` lines in another.

In [ ]:
import inspect
print(inspect.getsource(draw_skewed_grid))

## Deep 6: Write the argument

"With a basis, every spot has exactly one address." You have seen it. Now explain why it **must** be true.

Hint: suppose a spot had two different addresses. Subtract one trip from the other. What kind of trip do you get, and why is that impossible for a basis?